# Researcher · Extractor · crawl4ai — test bench

Verifies the Kaizen **Researcher** (arXiv + broad web + GitHub) and **Extractor**
(crawl4ai → full text **and images** → temporary RAG). Run top-to-bottom; each
section prints what worked so you can see exactly where (if anywhere) it fails.

If a step returns nothing it's almost always **network**: arXiv 301/blocked, or
no internet. crawl4ai needs a browser once: `.venv/bin/python -m playwright install chromium`.
Disable web crawling with `KAIZEN_USE_CRAWL4AI=0`; tune timeouts with `KAIZEN_NET_TIMEOUT`.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath("../../src"))
from gelochip.kaizen import researcher as R
QUERY = "cascode amplifier 15dB 50MHz gf180 cmos layout"
print("query:", QUERY)

## 1 · arXiv search  (httpx → arXiv API, https + follow-redirects)

In [ ]:
papers = R.search_arxiv("cascode amplifier cmos analog", max_results=4)
print(f"{len(papers)} papers")
for p in papers:
    print(" •", p["title"][:80]); print("   ", p["url"])

## 2 · Web + GitHub search  (ddgs — google-like, no API key)

In [ ]:
web = R.search_web(QUERY, max_results=4)
gh  = R.search_web("glayout cascode amplifier", max_results=3, github=True)
print(f"web={len(web)}  github={len(gh)}")
for r in (web + gh):
    print(f" • [{r['source']}] {r['title'][:70]}  ->  {r['url']}")

## 3 · crawl4ai — extract full TEXT + IMAGES from a page

In [ ]:
urls = [r["url"] for r in web[:2] if r.get("url")]
print("crawling:", urls)
t = time.time()
pages = R.crawl_pages(urls, max_chars=800, max_imgs=4)
print(f"crawled {len(pages)} pages in {time.time()-t:.1f}s")
for u, pg in pages.items():
    print("\n===", u, "===")
    print("TEXT:", (pg["text"] or "")[:300].replace("\n", " "), "…")
    print("IMAGES:", pg["images"])

## 4 · Full research()  — what the agent grounds on (text + images)

In [ ]:
t = time.time()
docs = R.research(QUERY, want=8)
from collections import Counter
print(f"{len(docs)} docs in {time.time()-t:.1f}s  by source:", dict(Counter(d['source'] for d in docs)))
for d in docs:
    print(f" • [{d['source']:6}] {d['title'][:70]}")

# show any extracted images inline
from IPython.display import Image, display
imgs = [d["url"] for d in docs if d["source"] == "image"]
print(f"\n{len(imgs)} extracted image(s):")
for u in imgs[:4]:
    try: display(Image(url=u, width=320))
    except Exception as e: print("  (could not render)", u)

## 5 · Extractor → temporary RAG, then query it

In [ ]:
job = "nbtest"
R.build_temp_rag(docs, job)
hits = R.query_temp_rag(job, "cascode gain bandwidth bias", k=3)
print(f"temp RAG returned {len(hits)} chunks:")
for h in hits:
    print(" •", f"[{h.metadata.get('source')}]", h.page_content[:120].replace("\n"," "), "…")
R.drop_temp_rag(job)
print("temp RAG dropped (it is per-job; promoted to permanent ChromaDB only on a DRC-clean result).")

## Notes
- The Studio **Prompt → GDSII** flow runs this automatically as the `research`
  node and shows the sources/images in the **Knowledge & Research** panel —
  so you can verify the agent's info *before* it generates code.
- On a DRC-clean result, the research + verified code are promoted into the
  permanent `rf_theory` / `glayout_knowledge` collections.